In [2]:
!pip install -U gdown torch_geometric transformers --quiet

import os
import gdown
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 92.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 92.6 MB/s eta 0:00:00:00:01


In [3]:
import gdown
import os
working_dir = '/kaggle/working/SMTPD/data_source'
os.makedirs(working_dir, exist_ok=True)
csv_id = '1tGzALqPqvyjmO6OazievOmaEt0B2OA3d'
gdown.download(f'https://drive.google.com/uc?id={csv_id}', os.path.join(working_dir, 'basic_view_pn_30.csv'), quiet=True)

'/kaggle/working/SMTPD/data_source/basic_view_pn_30.csv'

In [ ]:
!pip install -U torch==2.1.0 torchvision==0.16.0 --quiet

In [1]:
import torch
torch_v = torch.__version__.split("+")[0]
cuda_v = "cu" + torch.version.cuda.replace(".", "")
url = f"https://data.pyg.org/whl/torch-{torch_v}+{cuda_v}.html"

!pip install -U torch-scatter torch-sparse torch-cluster torch-spline-conv -f {url}

torch: 2.10.0+cu128
cuda: 12.8
Looking in links: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 53.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 89.6 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 93.3 MB/s eta 0:00:00:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.0 MB/s eta 0:00:00


In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
from transformers import BertTokenizer, BertModel
from torch_geometric.nn import GATConv, knn_graph
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

CSV_PATH = '/kaggle/working/SMTPD/data_source/basic_view_pn_30.csv'
IMG_DIR = '/kaggle/input/datasets/nguynnhnthiu/img-yt-zip/img_yt'
BATCH_SIZE = 64

TEXT_DIM = 768
VIS_DIM = 2048

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"CSV file not found at {CSV_PATH}")

resnet = models.resnet101(pretrained=True).to(device).eval()
resnet = torch.nn.Sequential(*(list(resnet.children())[:-1]))

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')
bert_model = BertModel.from_pretrained('bert-base-multilingual-cased').to(device).eval()

def get_day_columns(frame):
    day_cols = []
    for col in frame.columns:
        if not str(col).startswith('Day '):
            continue
        try:
            day_num = int(str(col).split(' ')[1])
        except (IndexError, ValueError):
            continue
        day_cols.append((day_num, col))
    day_cols.sort(key=lambda x: x[0])
    return day_cols

def extract_features(batch_df, day_cols, max_day):
    batch_features = []
    batch_targets = []
    for _, row in batch_df.iterrows():
        inputs = tokenizer(str(row['title']), return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
        with torch.no_grad():
            txt_feat = bert_model(**inputs).last_hidden_state[:, 0, :].cpu().numpy().flatten()

        path = os.path.join(IMG_DIR, f"{row['video_id']}.jpg")
        if os.path.exists(path):
            try:
                img = Image.open(path).convert('RGB')
                prep = transforms.Compose([
                    transforms.Resize(256),
                    transforms.CenterCrop(224),
                    transforms.ToTensor(),
                    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
                ])
                vis_feat = resnet(prep(img).unsqueeze(0).to(device)).cpu().numpy().flatten()
            except Exception:
                vis_feat = np.zeros(VIS_DIM)
        else:
            vis_feat = np.zeros(VIS_DIM)

        day1 = row.get('Day 1', np.nan)
        if pd.isna(day1):
            continue

        for day_num, col_name in day_cols:
            if day_num == 1:
                continue
            target_val = row.get(col_name, np.nan)
            if pd.isna(target_val):
                continue
            t_target = float(day_num) / float(max_day)
            combined = np.concatenate([txt_feat, vis_feat, [day1], [t_target]])
            batch_features.append(combined)
            batch_targets.append(target_val)
    return np.array(batch_features), np.array(batch_targets)

class HybridGraphCPDN(torch.nn.Module):
    def __init__(self, feat_dim=2817, hid_ch=128, k_neighbors=3):
        super().__init__()
        self.k = k_neighbors

        self.base_regressor = torch.nn.Sequential(
            torch.nn.Linear(2, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1)
        )

        self.proj = torch.nn.Sequential(
            torch.nn.Linear(feat_dim, hid_ch),
            torch.nn.LayerNorm(hid_ch),
            torch.nn.ReLU()
        )

        self.conv1 = GATConv(hid_ch, hid_ch, heads=4, concat=True)
        self.conv2 = GATConv(hid_ch * 4, hid_ch, heads=1, concat=False)
        self.gnn_out = torch.nn.Linear(hid_ch, 1)

    def forward(self, x):
        content_features = x[:, :-2]
        early_pop = x[:, -2:-1]
        t_target = x[:, -1:]

        base_in = torch.cat([early_pop, t_target], dim=1)
        base_pred = early_pop.squeeze(-1) + self.base_regressor(base_in).squeeze(-1)

        content_with_t = torch.cat([content_features, t_target], dim=1)
        h = self.proj(content_with_t)
        batch_k = min(self.k, h.shape[0] - 1)
        if batch_k > 0:
            edge_index = knn_graph(h, k=batch_k, loop=True)
        else:
            edge_index = torch.empty((2, 0), dtype=torch.long, device=h.device)

        h = self.conv1(h, edge_index)
        h = F.elu(h)
        h = self.conv2(h, edge_index)
        h = F.elu(h)

        graph_refinement = self.gnn_out(h).squeeze(-1)

        return base_pred + graph_refinement

df = pd.read_csv(CSV_PATH).reset_index(drop=True)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

day_cols = get_day_columns(df)
day_nums = [n for n, _ in day_cols]
if 1 not in day_nums:
    raise ValueError("Dataset must contain 'Day 1' as the early signal input.")
max_day = max(day_nums)

feat_dim_content = TEXT_DIM + VIS_DIM + 1
model = HybridGraphCPDN(feat_dim=feat_dim_content, hid_ch=128, k_neighbors=3).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

epochs = 15
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0
    train_df = train_df.sample(frac=1).reset_index(drop=True)

    num_batches = int(np.ceil(len(train_df) / BATCH_SIZE))
    for b_idx in tqdm(range(num_batches), desc=f"Epoch {epoch}/{epochs}"):
        batch_data = train_df.iloc[b_idx*BATCH_SIZE : (b_idx+1)*BATCH_SIZE]

        x_np, y_np = extract_features(batch_data, day_cols, max_day)
        if len(x_np) == 0:
            continue

        x = torch.tensor(x_np, dtype=torch.float).to(device)
        y = torch.tensor(y_np, dtype=torch.float).to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = F.l1_loss(out, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    model.eval()
    test_batch = test_df.sample(n=min(512, len(test_df)), random_state=42)
    x_val_np, y_val_np = extract_features(test_batch, day_cols, max_day)

    x_val = torch.tensor(x_val_np, dtype=torch.float).to(device)
    y_val = torch.tensor(y_val_np, dtype=torch.float).to(device)

    with torch.no_grad():
        val_out = model(x_val)
        mae = mean_absolute_error(y_val.cpu().numpy(), val_out.cpu().numpy())

    print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss/num_batches:.4f} | Val MAE: {mae:.4f}")

    scheduler.step(mae)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet101_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet101_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet101-63fe2227.pth" to /root/.cache/torch/hub/checkpoints/resnet101-63fe2227.pth


100%|██████████| 171M/171M [00:00<00:00, 217MB/s] 


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

...

Epoch 9/15: 100%|██████████| 1060/1060 [33:43<00:00,  1.91s/it]


Epoch 09 | Train Loss: 0.7599 | Val MAE: 0.7743


Epoch 10/15: 100%|██████████| 1060/1060 [33:53<00:00,  1.92s/it]


Epoch 10 | Train Loss: 0.7578 | Val MAE: 0.7783


Epoch 11/15:  79%|███████▉  | 838/1060 [26:43<06:59,  1.89s/it]